# NIfTI scans to training slices

Builds the real-scan dataset used by `train.py`:

```
data_real/Training/<class>/<patientID>_z###.jpg
data_real/Testing/<class>/<patientID>_z###.jpg
```

Each patient goes entirely into Training or Testing (20% of patients are held out), so test accuracy is not inflated by near-identical slices of the same patient. The conversion code lives in `nifti_slices.py` in the repo; this notebook only points it at the Kaggle datasets and zips the result for Colab/Drive.

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count(os.sep) - 2
    if depth > 3:
        continue
    print("  " * depth + os.path.basename(root) + f"/  ({len(dirs)} folders, {len(files)} files)")

In [ ]:
# Get nifti_slices.py from the repo (needs Internet ON in the notebook settings).
# If the repo is private, upload nifti_slices.py as a Kaggle dataset instead and
# set SCRIPT to its path under /kaggle/input.
BRANCH = "main"
!rm -rf /kaggle/working/repo
!git clone --depth 1 -b {BRANCH} https://github.com/imfonggg/AI-for-Brain-Tumor-Detection.git /kaggle/working/repo
!pip install -q nibabel
SCRIPT = "/kaggle/working/repo/nifti_slices.py"

In [ ]:
# Paths to datasets attached via "Add Data" (check the right-hand panel for the exact path)
GLIOMA_ROOT = "/kaggle/input/datasets/vietanh21/brats-2021-task-1-dataset"
MENINGIOMA_ROOT = "/kaggle/input/datasets/dukhailly/meningioma"
OUT_DIR = "/kaggle/working/data_real"

# Glioma uses only the UCSF-PDGM cases inside BraTS 2021 (add --all_brats for every case).
# notumor comes from IXI healthy controls, streamed from the IXI server.
# --notumor_from_tumor_patients also adds tumor-free slices from the glioma/meningioma
# scans; see the note in nifti_slices.py before turning it on.
!rm -rf {OUT_DIR}
!python {SCRIPT} --out_dir {OUT_DIR} \
    --glioma_root {GLIOMA_ROOT} \
    --meningioma_root {MENINGIOMA_ROOT} \
    --ixi \
    --slices_per_patient 8 --max_patients 400

In [ ]:
# Quick visual check: 6 random slices per class from the training split
import glob, os, random
import matplotlib.pyplot as plt
from PIL import Image

classes = sorted(os.listdir(f"{OUT_DIR}/Training"))
fig, axes = plt.subplots(len(classes), 6, figsize=(15, 2.7 * len(classes)), squeeze=False)
for row, cls in enumerate(classes):
    files = glob.glob(f"{OUT_DIR}/Training/{cls}/*.jpg")
    for col, f in enumerate(random.sample(files, min(6, len(files)))):
        axes[row, col].imshow(Image.open(f), cmap="gray")
        axes[row, col].set_title(f"{cls}\n{os.path.basename(f)[:22]}", fontsize=7)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Zip for download -> upload data_real.zip to Google Drive for the Colab training notebook
import shutil
shutil.make_archive("/kaggle/working/data_real", "zip", OUT_DIR)
print("Saved /kaggle/working/data_real.zip")